# Validación del pipeline (Parcial MCD-402)

Este notebook ejecuta **las mismas tareas que el DAG de Airflow** (`src/pipeline.py`), una tras otra, sin Docker:

`validar_datos → preprocesar_datos → optimizar_hiperparametros → entrenar_modelos → backtesting → seleccionar_campeon → registrar_experimentos → generar_reporte`

Sirve para revisar el flujo en Google Colab o en tu máquina. Como en Colab no hay servidor de MLflow, los runs se guardan en `mlflow_local.db` (respaldo definido en `config.yaml → tracking.tracking_uri_respaldo`).

In [ ]:
# Solo en Google Colab: montar Drive (en tu maquina, salta esta celda)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

In [ ]:
import os, sys

if EN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/MLOps/automl_venta_financiera'  # <-- editar
else:
    PROJECT_ROOT = os.path.abspath('..')  # el notebook esta en notebooks/

DATASET_PATH = 'data/input'   # o 'data/muestra' para una prueba rapida
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
os.environ['PROYECTO_DIR'] = PROJECT_ROOT
print('Proyecto en:', PROJECT_ROOT)

In [ ]:
# Librerias (en Colab). En tu maquina: pip install -r requirements.txt
if EN_COLAB:
    !pip install -q catboost lightgbm optuna mlflow psutil PyYAML

In [ ]:
from datetime import datetime
from src import pipeline

dag_run_id = 'notebook__' + datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
carpeta = None
for tarea in pipeline.ORDEN_TAREAS:
    salida = pipeline.ejecutar_tarea(tarea, dag_run_id, carpeta_corrida=carpeta,
                                     overrides={'input_path': DATASET_PATH})
    carpeta = salida['carpeta_corrida']
print('Corrida:', carpeta)

In [ ]:
import json
import pandas as pd

with open(os.path.join(carpeta, 'results_summary.json'), encoding='utf-8') as f:
    resumen = json.load(f)

print('Campeon:', resumen['campeon'], '| estado:', resumen['seleccion']['estado'])
print(resumen['seleccion']['motivo'])
pd.DataFrame(resumen['tabla_modelos'])

In [ ]:
# Backtesting del campeon (meses posteriores al corte, sin reentrenar)
pd.DataFrame(resumen['backtesting_campeon']['waves'])

In [ ]:
# Logs de computo por modelo y por tarea
display(pd.DataFrame(resumen['compute_logs']['modelos']).T[['duracion_s', 'cpu_pct_prom', 'cpu_pct_max',
                                                           'ram_mb_prom', 'ram_mb_max', 'modo_computo', 'tamano_modelo_mb']])
pd.DataFrame(resumen['compute_logs']['tareas']).T[['duracion_s', 'cpu_pct_prom', 'cpu_pct_max', 'ram_mb_max']]

In [ ]:
# Por que se descarto cada modelo en GetBestModel()
pd.Series(resumen['seleccion']['descartados'], name='motivo').to_frame()